[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:

import torch
import torch.nn as nn
import math

In [37]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model, self.num_heads = d_model, num_heads
        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        q = torch.stack(self.W_q(x_q).chunk(self.num_heads, dim=-1), dim=1)
        k = torch.stack(self.W_k(x_kv).chunk(self.num_heads, dim=-1),dim=1)
        v = torch.stack(self.W_v(x_kv).chunk(self.num_heads, dim=-1),dim=1)
        dh = self.d_model//self.num_heads
        # print(q.shape, k.shape, v.shape,)
        s = torch.einsum('ijkl,ijml->ijkm', q, k) / math.sqrt(dh)
        # print(s.shape)
        attn = torch.softmax(s, dim=-1)
        head = torch.einsum('bhoi,bhiv->bhov', attn, v)
        # print(head.shape)
        head = head.reshape(head.shape[0], head.shape[2], head.shape[1]*head.shape[3])
        return head
        # pass  # Q from x_q, K/V from x_kv, no causal mask

In [48]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model, self.num_heads = d_model, num_heads
        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        # q_tuple = self.W_q(x_q).chunk(self.num_heads, dim=-1)
        #True # print(torch.equal(torch.stack(q_tuple, dim=1), torch.stack(q_tuple, dim=-2).transpose(1,2)))
        q = torch.stack(self.W_q(x_q).chunk(self.num_heads, dim=-1), dim=1)
        k = torch.stack(self.W_k(x_kv).chunk(self.num_heads, dim=-1),dim=1)
        v = torch.stack(self.W_v(x_kv).chunk(self.num_heads, dim=-1),dim=1)
        dh = self.d_model//self.num_heads
        # print(q.shape, k.shape, v.shape,)
        s = torch.einsum('ijkl,ijml->ijkm', q, k) / math.sqrt(dh)
        # print(s.shape)
        attn = torch.softmax(s, dim=-1)
        head = torch.einsum('bhoi,bhiv->bhov', attn, v)
        # print(head.shape)
        head = head.transpose(1,2).reshape(head.shape[0], head.shape[2], head.shape[1]*head.shape[3])
        output = self.W_o(head)
        return output
        # pass  # Q from x_q, K/V from x_kv, no causal mask

In [49]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [50]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (5.8ms)
  ✅ [2/4] Q and KV different lengths (1.8ms)
  ✅ [3/4] No causal mask — all KV affects all Q (9.3ms)
  ✅ [4/4] Gradient flow (3.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (20.3ms total)
  Progress saved. Run status() to see your dashboard.



In [56]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


In [57]:
def naive_reference(m, x_q, x_kv):
    B, Sq, D = x_q.shape
    Skv = x_kv.shape[1]
    H = m.num_heads
    dh = D // H

    Q = x_q @ m.W_q.weight.T + m.W_q.bias        # (B, Sq,  D)
    K = x_kv @ m.W_k.weight.T + m.W_k.bias       # (B, Skv, D)
    V = x_kv @ m.W_v.weight.T + m.W_v.bias       # (B, Skv, D)

    out = torch.zeros(B, Sq, D, dtype=x_q.dtype)
    for b in range(B):
        for h in range(H):
            sl = slice(h * dh, (h + 1) * dh)     # this head's feature block
            Qh, Kh, Vh = Q[b, :, sl], K[b, :, sl], V[b, :, sl]

            S = torch.zeros(Sq, Skv, dtype=x_q.dtype)
            for i in range(Sq):
                for j in range(Skv):
                    S[i, j] = torch.dot(Qh[i], Kh[j]) / math.sqrt(dh)

            A = torch.softmax(S, dim=-1)         # rows sum to 1 over Skv
            out[b, :, sl] = A @ Vh               # (Sq, dh)

    return out @ m.W_o.weight.T + m.W_o.bias


In [58]:
def sdpa_reference(m, x_q, x_kv):
    B, Sq, D = x_q.shape
    Skv = x_kv.shape[1]
    H = m.num_heads
    dh = D // H
    q = m.W_q(x_q).view(B, Sq, H, dh).transpose(1, 2)
    k = m.W_k(x_kv).view(B, Skv, H, dh).transpose(1, 2)
    v = m.W_v(x_kv).view(B, Skv, H, dh).transpose(1, 2)
    o = F.scaled_dot_product_attention(q, k, v)          # no mask
    return m.W_o(o.transpose(1, 2).reshape(B, Sq, D))


In [59]:
def official_reference(m, x_q, x_kv):
    D, H = m.d_model, m.num_heads
    mha = nn.MultiheadAttention(D, H, batch_first=True, bias=True)
    with torch.no_grad():
        mha.in_proj_weight.copy_(
            torch.cat([m.W_q.weight, m.W_k.weight, m.W_v.weight], dim=0))
        mha.in_proj_bias.copy_(
            torch.cat([m.W_q.bias, m.W_k.bias, m.W_v.bias], dim=0))
        mha.out_proj.weight.copy_(m.W_o.weight)
        mha.out_proj.bias.copy_(m.W_o.bias)
    out, _ = mha(x_q, x_kv, x_kv, need_weights=False)     # Q from decoder
    return out


In [60]:
def run():
    torch.manual_seed(0)
    B, Sq, Skv, D, H = 2, 6, 10, 64, 4
    m = MultiHeadCrossAttention(D, H).double()
    x_q = torch.randn(B, Sq, D, dtype=torch.float64)
    x_kv = torch.randn(B, Skv, D, dtype=torch.float64)

    results = []

    def check(name, cond, detail=""):
        results.append((name, cond, detail))
        print(f"  {'PASS' if cond else 'FAIL'}  {name}" + (f"   {detail}" if detail else ""))

    print("Cross-attention verification\n" + "-" * 58)

    # 1. output length is Sq, not Skv
    out = m(x_q, x_kv)
    check("output shape == (B, S_q, D), not S_kv",
          out.shape == (B, Sq, D), f"got {tuple(out.shape)}, S_kv={Skv}")

    # 2. matches the equations transcribed as explicit loops
    d = (out - naive_reference(m, x_q, x_kv)).abs().max().item()
    check("matches naive per-head loop reference", d < 1e-10, f"max diff {d:.3e}")

    # 3. matches PyTorch's fused kernel
    d = (out - sdpa_reference(m, x_q, x_kv)).abs().max().item()
    check("matches F.scaled_dot_product_attention", d < 1e-10, f"max diff {d:.3e}")

    # 4. matches nn.MultiheadAttention
    mf = MultiHeadCrossAttention(D, H)
    with torch.no_grad():
        for a, b in zip(mf.parameters(), m.parameters()):
            a.copy_(b.float())
    xqf, xkvf = x_q.float(), x_kv.float()
    d = (mf(xqf, xkvf) - official_reference(mf, xqf, xkvf)).abs().max().item()
    check("matches nn.MultiheadAttention", d < 1e-5, f"max diff {d:.3e}")

    # 5. attention rows are distributions over the SOURCE positions
    q = torch.stack(m.W_q(x_q).chunk(H, -1), 1)
    k = torch.stack(m.W_k(x_kv).chunk(H, -1), 1)
    A = torch.softmax(torch.einsum('ijkl,ijml->ijkm', q, k) / math.sqrt(D // H), -1)
    check("attn shape is rectangular (B,H,S_q,S_kv)",
          A.shape == (B, H, Sq, Skv), f"got {tuple(A.shape)}")
    check("attn rows sum to 1 over S_kv",
          torch.allclose(A.sum(-1), torch.ones(B, H, Sq, dtype=A.dtype)))
    check("attn is non-causal (no zeros in upper triangle)",
          bool((A > 0).all()), "every query sees every source position")

    # 6. permuting source positions permutes attention but leaves output
    #    invariant -- attention is order-agnostic without positional encoding
    perm = torch.randperm(Skv)
    check("output invariant to source permutation",
          torch.allclose(m(x_q, x_kv[:, perm]), out, atol=1e-10))

    # 7. degenerates to self-attention when the two streams coincide
    o_self = m(x_q, x_q)
    q2 = torch.stack(m.W_q(x_q).chunk(H, -1), 1)
    k2 = torch.stack(m.W_k(x_q).chunk(H, -1), 1)
    v2 = torch.stack(m.W_v(x_q).chunk(H, -1), 1)
    A2 = torch.softmax(torch.einsum('ijkl,ijml->ijkm', q2, k2) / math.sqrt(D // H), -1)
    h2 = torch.einsum('bhij,bhjd->bhid', A2, v2).transpose(1, 2).reshape(B, Sq, D)
    check("reduces to unmasked self-attention when x_q == x_kv",
          torch.allclose(o_self, m.W_o(h2), atol=1e-10))

    # 8. gradient reaches BOTH inputs (the encoder path is the point)
    a = x_q.clone().requires_grad_()
    b = x_kv.clone().requires_grad_()
    m(a, b).sum().backward()
    check("gradient flows to x_q", a.grad is not None and a.grad.abs().sum() > 0)
    check("gradient flows to x_kv (decoder loss -> encoder)",
          b.grad is not None and b.grad.abs().sum() > 0)

    # 9. scaling is actually present: without 1/sqrt(dh) the softmax
    #    saturates. Large-magnitude inputs should not collapse to one-hot.
    big = x_kv * 10
    A_big = torch.softmax(
        torch.einsum('ijkl,ijml->ijkm', q,
                     torch.stack(m.W_k(big).chunk(H, -1), 1)) / math.sqrt(D // H), -1)
    A_unscaled = torch.softmax(
        torch.einsum('ijkl,ijml->ijkm', q,
                     torch.stack(m.W_k(big).chunk(H, -1), 1)), -1)
    check("1/sqrt(d_h) present (entropy > unscaled)",
          A_big.max().item() < A_unscaled.max().item(),
          f"peak {A_big.max():.4f} vs unscaled {A_unscaled.max():.4f}")

    # 10. head count actually matters (guards against a silent merge bug)
    m2 = MultiHeadCrossAttention(D, 8).double()
    with torch.no_grad():
        for a_, b_ in zip(m2.parameters(), m.parameters()):
            a_.copy_(b_)
    check("H=4 and H=8 give different outputs (heads are real)",
          not torch.allclose(m2(x_q, x_kv), out, atol=1e-6))

    print("-" * 58)
    n = sum(1 for _, c, _ in results if c)
    print(f"{n}/{len(results)} passed" + ("" if n == len(results) else "  <-- see FAILs above"))


In [61]:
if __name__ == "__main__":
    run()


Cross-attention verification
----------------------------------------------------------
  PASS  output shape == (B, S_q, D), not S_kv   got (2, 6, 64), S_kv=10
  PASS  matches naive per-head loop reference   max diff 1.388e-16
  PASS  matches F.scaled_dot_product_attention   max diff 1.388e-16
  PASS  matches nn.MultiheadAttention   max diff 8.941e-08
  PASS  attn shape is rectangular (B,H,S_q,S_kv)   got (2, 4, 6, 10)
  PASS  attn rows sum to 1 over S_kv
  PASS  attn is non-causal (no zeros in upper triangle)   every query sees every source position
  PASS  output invariant to source permutation
  PASS  reduces to unmasked self-attention when x_q == x_kv
  PASS  gradient flows to x_q
  PASS  gradient flows to x_kv (decoder loss -> encoder)
  PASS  1/sqrt(d_h) present (entropy > unscaled)   peak 0.9991 vs unscaled 1.0000
  PASS  H=4 and H=8 give different outputs (heads are real)
----------------------------------------------------------
13/13 passed
